## 1. 설정

In [1]:
SCOPETYPE = "OPENADC"          # CW-Lite / CW-Pro / Husky
PLATFORM  = "CW308_STM32F4"    # CW308T-STM32F4HWC (STM32F415)도 동일
SS_VER    = "SS_VER_2_1"         # 펌웨어는 SimpleSerial 2.1만 지원 (Setup에서 target = SimpleSerial2)

PARAMS     = 768               # 512 | 768 | 1024
IMPL       = "clean"           # "m4fstack" | "m4fspeed" | "clean" — 트리거(A/B)는 clean에만 있음
TRIGGER_A  = 1                 # 1: 비밀키 basemul(polyvec_basemul_acc_montgomery)에서 트리거 / 0: 끔
TRIGGER_B  = 0                # 1: poly_tomsg(메시지 변환)에서 트리거 / 0: 끔 — A/B 독립적으로 켜고 끌 수 있음
OPT        = "3"               # 컴파일 최적화: "0" = -O0 (feasibility 단계, 누출 명확성 우선) / "3" = -O3 (pqm4 기본)
N_TRACES   = 1000              # 수집할 파형 수

In [2]:
%run "Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.adc.trig_count                     changed from 162946                    to 13425820                 
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx         

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER" "$PARAMS" "$IMPL" "$TRIGGER_A" "$TRIGGER_B" "$OPT"
cd ../firmware/mcu/simpleserial-mlkem                                   # 펌웨어 빌드 디렉토리로 이동
make PLATFORM=$1 SS_VER=$2 PARAMS=$3 IMPL=$4 TRIGGER_A=$5 TRIGGER_B=$6 OPT=$7 -j   # 설정대로 컴파일(-j 병렬)

In [ ]:
fw_path = "../firmware/mcu/simpleserial-mlkem/simpleserial-mlkem{}-{}-{}.hex".format(  # 빌드된 hex 경로
    PARAMS, IMPL, PLATFORM)                      # 파일명에 파라미터/구현/플랫폼이 들어감
cw.program_target(scope, prog, fw_path)          # STM32에 펌웨어 플래시
reset_target(scope)                              # 보드 리셋(nRST): RAM 초기화 후 펌웨어 재시작

### 검증용 파이썬 MLKEM 라이브러리 import

In [ ]:
from kyber_py.ml_kem import ML_KEM_512, ML_KEM_768, ML_KEM_1024   # PC 기준 구현

KEM = {512: ML_KEM_512, 768: ML_KEM_768, 1024: ML_KEM_1024}[PARAMS]   # PARAMS에 맞는 KEM 선택

### 트레이스 캡쳐 관련 함수 정의

In [ ]:
target.simpleserial_write("i", bytearray())      # 펌웨어 정보 요청 'i'
fw = target.simpleserial_read("r", 4, timeout=1000)   # 응답 4바이트
if fw is None:                                   # 없으면
    raise RuntimeError("'i' 응답 없음: 보드 리셋/통신/SS_VER 확인")

impl_name = {1: "m4fstack", 2: "m4fspeed", 3: "clean"}   # fw[1] -> 구현명
fw_params, fw_impl, fw_trigger = fw[0] * 256, impl_name.get(fw[1]), fw[2]   # 응답 해석
print("firmware:", fw_params, fw_impl, f"trigger=0b{fw_trigger:02b} (bit0=A bit1=B)", "chunk", fw[3])

# 설정 셀 값으로 기대 트리거 비트 생성 -> 엉뚱한 hex를 플래시한 채로 수집하는 실수 방지
want_trigger = (1 if TRIGGER_A else 0) | (2 if TRIGGER_B else 0)   # bit0=A, bit1=B
assert (fw_params, fw_impl, fw_trigger) == (PARAMS, IMPL, want_trigger), "플래시된 펌웨어가 설정과 다릅니다"

# SimpleSerial 2.1 패킷 하나에 데이터는 최대 249B라 dk/ct를 조각으로 보냄
CHUNK = fw[3]                                     # 조각 크기(펌웨어가 알려준 240)

def send_chunks(cmd, data):                       # data를 CHUNK 단위로 전송
    """data를 CHUNK 바이트씩 잘라 보냄: scmd = 조각 번호, 마지막 조각은 남은 길이 그대로 (패딩 없음)"""
    for idx, i in enumerate(range(0, len(data), CHUNK)):   # 조각 번호 idx
        target.send_cmd(cmd, idx, data[i:i + CHUNK])       # (idx*CHUNK) 위치에 memcpy
        err = target.simpleserial_wait_ack(timeout=500)    # 조각 ack
        if err is None:                                    # 응답 없으면
            raise RuntimeError(f"'{cmd}' 조각 {idx}: ack 없음")
        if err[0] != 0:                                    # 에러 코드면
            raise RuntimeError(f"'{cmd}' 조각 {idx}: 펌웨어 에러 0x{err[0]:02x}")

def load_key(dk):                                 # 비밀키 적재
    """비밀키(dk, FIPS 203 바이트 형식)를 보드 RAM에 적재.
    정적 키 모델: 한 번 올리고 같은 키로 decaps를 반복."""
    send_chunks("s", dk)                          # 's' 명령

def decaps(ct, capture=False):                    # 암호문 보내고 decaps
    """암호문을 보내고 보드에서 crypto_kem_dec 실행 -> 공유비밀 32B 반환.
    capture=True면 그 decaps 동안의 파형을 수집."""
    send_chunks("c", ct)                         # 1) 암호문 적재
    if capture:                                  #    수집 모드면
        scope.arm()                              # 2) 'd' 전에 arm (트리거 안 놓침)
    target.simpleserial_write("d", bytearray())  # 3) 복호화 시작 (A/B 트리거 발생)
    if capture and scope.capture():              # 4) capture() 타임아웃이면 True
        print("경고: 캡처 타임아웃")
    ss = target.simpleserial_read("r", 32, timeout=3000)  # 5) 공유비밀 32B
    return None if ss is None else bytes(ss)     # 없으면 None

### 인터랙티브 플롯

In [ ]:
from bokeh.plotting import figure, show       # Bokeh: 인터랙티브 플롯 라이브러리
from bokeh.io import output_notebook           # 그림을 노트북 안에 표시
from bokeh.resources import INLINE             # Bokeh JS를 노트북에 인라인 삽입(오프라인에서도 됨)
from bokeh.palettes import Category10_10       # 선 색 10개
import numpy as np                             # 배열

output_notebook(INLINE)                        # 노트북 출력 모드 켜기

### MLKEM 복호화 트레이스 TVLA 수집

In [ ]:
import numpy as np                              # 배열/저장
import pathlib, time                            # 경로/타임스탬프
from tqdm.notebook import trange                # 진행바

PARAMS = 768                     # 저장 이름용 (설정 셀과 동일하게)
IMPL   = "clean"                 # 저장 이름용 구현
N = 4000                         # 총 파형 수 (class당 ~2000)
rng = np.random.default_rng(0)   # 재현 가능한 난수 (class 선택용)

ek, dk = KEM.keygen()            # 고정 키 쌍 생성
ss_ref0, ct0 = KEM.encaps(ek)    # 고정 class가 매번 쓸 암호문 (한 번만)
load_key(dk)                     # 비밀키 고정, 한 번만 적재

waves, labels, cts = [], [], []  # 파형/라벨/암호문 누적
for _ in trange(N):              # N번 수집
    fixed = rng.integers(2)      # ★ 매 파형 무작위로 class 선택 (드리프트가 t로 새는 것 방지)
    if fixed:                    # class 1 = fixed
        ct, ss_ref = ct0, ss_ref0            # 고정 암호문 재사용
    else:                        # class 0 = random
        ss_ref, ct = KEM.encaps(ek)          # 매번 새 암호문
    ss = decaps(ct, capture=True)            # 보드 decaps + 파형
    if ss != ss_ref:             # 통신 오류 파형은 버림
        continue
    waves.append(scope.get_last_trace())     # 파형 저장
    labels.append(int(fixed))                # class 라벨 저장
    cts.append(np.frombuffer(ct, dtype=np.uint8))   # ★ accept된 파형에만, waves와 정렬

waves  = np.array(waves, dtype=np.float32)   # (M, S) 파형
labels = np.array(labels, dtype=np.uint8)    # (M,) 라벨
cts    = np.array(cts)                        # (M, 1088), waves[i] <-> cts[i]

out = pathlib.Path("../traces"); out.mkdir(exist_ok=True)   # 저장 폴더
stamp = time.strftime("%Y%m%d-%H%M%S")                      # 타임스탬프
path = out / f"mlkem{PARAMS}-{IMPL}-tvla-{stamp}.npz"       # tvla 파일명
np.savez_compressed(                                        # 압축 저장
    path,
    waves=waves, labels=labels, ct=cts,                     # 파형+라벨+ct
    ct0=np.frombuffer(ct0, dtype=np.uint8),                 # 고정 암호문(메타)
     dk=np.frombuffer(dk, dtype=np.uint8),                  # 비밀키(재현용)
    ek=np.frombuffer(ek, dtype=np.uint8),                   # 공개키(재현용)
)
print(path)                                                 # 저장 경로
print(waves.shape, "fixed:", int((labels==1).sum()), "random:", int((labels==0).sum()))   # class 수

### TVLA용 트레이스 불러오기

In [ ]:
import numpy as np                              # 배열
import glob                                     # 파일 검색

path = sorted(glob.glob("../traces/mlkem*-tvla-*.npz"))[-1]   # 가장 최근 TVLA 파일
print("loading:", path)                         # 어떤 파일

with np.load(path) as d:                        # 열고 자동으로 닫기(윈도우 잠금 방지)
    waves  = d["waves"]      # (M, samples) float32 — 파형
    labels = d["labels"]     # (M,) uint8  — 1=fixed, 0=random

print(waves.shape, "fixed:", int((labels==1).sum()), "random:", int((labels==0).sum()))   # 개수 확인

### Welch t-test

In [ ]:
import numpy as np                              # 배열
import matplotlib.pyplot as plt                 # 그래프
from scipy.stats import ttest_ind               # Welch t-검정

def tvla_t(waves, labels):                      # 샘플별 Welch t 통계량
    group1 = waves[labels == 1]      # fixed class 파형
    group2 = waves[labels == 0]      # random class 파형
    return ttest_ind(group1, group2, axis=0, equal_var=False)[0]   # Welch(비등분산) t, [0]=통계량

t = tvla_t(waves, labels)                        # 전체 데이터 t

plt.figure(figsize=(16, 4))                      # 그림 크기
plt.plot(t, lw=0.6)                              # t 곡선
plt.axhline( 4.5, color='r', ls='--'); plt.axhline(-4.5, color='r', ls='--')   # ±4.5 임계선
plt.xlabel("sample"); plt.ylabel("t"); plt.title("TVLA fixed vs random ct")    # 라벨/제목
plt.show()                                       # 표시
print("max |t| =", float(np.nanmax(np.abs(t))))  # 최대 |t| (4.5 넘으면 누출)

### 거짓 양성 점검 — 반쪽 2개 독립 검정

In [ ]:
h = len(waves) // 2                              # 데이터 절반 지점
t1 = tvla_t(waves[:h], labels[:h])               # 앞 절반 t
t2 = tvla_t(waves[h:], labels[h:])               # 뒤 절반 t
both = (np.abs(t1) > 4.5) & (np.abs(t2) > 4.5)   # 두 반쪽 모두 같은 위치에서 넘어야 진짜 누출
print("두 반쪽 모두 누출로 잡힌 샘플 수:", int(both.sum()))   # 거짓양성 점검

plt.figure(figsize=(16, 4))                      # 그림
plt.plot(t1, lw=0.6, alpha=0.7, label="half 1")  # 앞 절반
plt.plot(t2, lw=0.6, alpha=0.7, label="half 2")  # 뒤 절반
plt.axhline(4.5, color='r', ls='--'); plt.axhline(-4.5, color='r', ls='--')   # 임계선
plt.xlabel("sample"); plt.ylabel("t"); plt.legend(); plt.title("TVLA half-split")   # 라벨/범례/제목
plt.show()                                       # 표시

|t| > 4.5인 구간이 누출. A 구간과 B 구간 각각에서 넘으면 "두 누출 공존"이 실증된 것.

In [ ]:
import numpy as np                              # 배열
from bokeh.plotting import figure, show         # 인터랙티브 플롯
from bokeh.io import output_notebook            # 노트북 출력
from bokeh.resources import INLINE              # JS 인라인
from bokeh.models import Span                   # 보조선(임계선)

output_notebook(INLINE)                          # 출력 모드 켜기

h = len(waves) // 2                              # 절반 지점
t1 = tvla_t(waves[:h], labels[:h])               # 앞 절반 t
t2 = tvla_t(waves[h:], labels[h:])               # 뒤 절반 t
both = (np.abs(t1) > 4.5) & (np.abs(t2) > 4.5)   # 둘 다 넘는 샘플

x = np.arange(len(t1))                           # x축(샘플 인덱스)
p = figure(width=1600, height=450, title="TVLA half-split (범례 클릭 = 숨기기/보이기)",   # 플롯 생성
           x_axis_label="sample", y_axis_label="t")

p.line(x, t1, line_color="navy",   legend_label="half 1")            # 앞 절반 선
p.line(x, t2, line_color="orange", legend_label="half 2", alpha=0.8) # 뒤 절반 선

for y in (4.5, -4.5):               # ±4.5 임계선
    p.add_layout(Span(location=y, dimension="width", line_color="red", line_dash="dashed"))

p.legend.click_policy = "hide"      # ★ 범례 클릭하면 그 선 숨김
show(p)                             # 표시

print("두 반쪽 모두 누출로 잡힌 샘플 수:", int(both.sum()))   # 거짓양성 점검